#test

In [ ]:
!unzip /content/testASL1.zip

Streaming output truncated to the last 5000 lines.
  inflating: asl_dataset 4/r/hand2_r_top_seg_3_cropped.jpeg  
  inflating: __MACOSX/asl_dataset 4/r/._hand2_r_top_seg_3_cropped.jpeg  
  inflating: asl_dataset 4/r/hand5_r_dif_seg_4_cropped.jpeg  
  inflating: __MACOSX/asl_dataset 4/r/._hand5_r_dif_seg_4_cropped.jpeg  
  inflating: asl_dataset 4/r/hand2_r_left_seg_4_cropped.jpeg  
  inflating: __MACOSX/asl_dataset 4/r/._hand2_r_left_seg_4_cropped.jpeg  
  inflating: asl_dataset 4/r/hand5_r_dif_seg_3_cropped.jpeg  
  inflating: __MACOSX/asl_dataset 4/r/._hand5_r_dif_seg_3_cropped.jpeg  
  inflating: asl_dataset 4/r/hand2_r_left_seg_3_cropped.jpeg  
  inflating: __MACOSX/asl_dataset 4/r/._hand2_r_left_seg_3_cropped.jpeg  
  inflating: asl_dataset 4/r/hand1_r_dif_seg_4_cropped.jpeg  
  inflating: __MACOSX/asl_dataset 4/r/._hand1_r_dif_seg_4_cropped.jpeg  
  inflating: asl_dataset 4/r/hand4_r_bot_seg_1_cropped.jpeg  
  inflating: __MACOSX/asl_dataset 4/r/._hand4_r_bot_seg_1_cropped.jpeg  


In [ ]:
# Verify the contents after unzipping
!ls -F
!ls -F asl_dataset

'asl_dataset 4'/   __MACOSX/   sample_data/   testASL1.zip
ls: cannot access 'asl_dataset': No such file or directory


#test asl

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing import image_dataset_from_directory

# 1. Load image datasets (assuming directory structure: asl_dataset/class_folder/images.jpg)
# The dataset contains 0-9 and a-z classes, totaling 36 classes.
image_size = (64, 64)
batch_size = 32
num_classes = 36  # Updated to 36 (0-9 + a-z)

# Load the full dataset and split into training and validation using validation_split
full_dataset = image_dataset_from_directory(
    'asl_dataset',  # Path to the main dataset directory
    image_size=image_size,
    batch_size=batch_size,
    label_mode='categorical',
    validation_split=0.2,  # 20% of data for validation
    subset='both',  # Create both training and validation subsets
    seed=42  # Ensure reproducibility
)

train_dataset = full_dataset[0]
validation_dataset = full_dataset[1]

# Confirm number of classes loaded
print(f"Number of classes detected: {len(train_dataset.class_names)}")
print("Class names:", train_dataset.class_names)

# Optional: Data augmentation to handle variations in hand positions/lighting
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

# 2. Building the Model (CNN for image classification)
def build_model(num_classes=num_classes):
    model = keras.Sequential([
        data_augmentation,  # Augmentation layer
        layers.Rescaling(1./255),  # Normalize pixel values to [0,1]

        # Convolutional layers for feature extraction
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=image_size + (3,)),  # RGB input
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),  # Flatten for dense layers
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.6),  # Prevent overfitting (increased from 0.5 to 0.6)
        layers.Dense(num_classes, activation='softmax')  # Softmax for 36-class output
    ])

    model.compile(optimizer='rmsprop',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# 3. Training the model
model = build_model(num_classes=num_classes)
history = model.fit(
    train_dataset,
    epochs=20,
    validation_data=validation_dataset,
    verbose=1
)

# 4. Evaluate accuracy and other metrics
results = model.evaluate(validation_dataset)
print(f"Test Loss: {results[0]}, Test Accuracy: {results[1]}")

# Optional: Save model for inference on new JPEGs
model.save('asl_model_36classes.h5')

# To predict on a new JPEG:
# new_image = tf.keras.preprocessing.image.load_img('path/to/new_hand_sign.jpg', target_size=(64, 64))
# new_image = tf.keras.preprocessing.image.img_to_array(new_image)
# new_image = tf.expand_dims(new_image, axis=0)  # Batch dimension
# prediction = model.predict(new_image)
# predicted_class = tf.argmax(prediction, axis=1)

First, let's list the contents of your current directory to see if `asl_dataset` appears:

In [ ]:
!ls -F

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#  Load  my data
df = pd.read_csv('heart.csv')

# 2. Data Preprocessing
X = df.drop('target', axis=1).values
y = df['target'].values

# Split  training and test sets
train_data, test_data, train_targets, test_targets = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Feature Normalization
mean = train_data.mean(axis=0)
train_data -= mean
std = train_data.std(axis=0)
train_data /= std

test_data -= mean
test_data /= std

#  Building  Model
def build_model():
    model = keras.Sequential([
        layers.Dense(128, activation="relu"),
        layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid") # Sigmoid for binary classification
    ])

    model.compile(optimizer="rmsprop",
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

# Validation set
x_val = train_data[:50]
partial_x_train = train_data[50:]
y_val = train_targets[:50]
partial_y_train = train_targets[50:]

#  Training the model
model = build_model()
history = model.fit(
    partial_x_train,
    partial_y_train,
    epochs=20,
    batch_size=16,
    validation_data=(x_val, y_val),
    verbose=1
)

#  Accuracy and other metrics
results = model.evaluate(test_data, test_targets)
print(f"Test Loss: {results[0]}, Test Accuracy: {results[1]}")

KeyboardInterrupt: 

If you don't see `asl_dataset` (or the folder where you expected it to be) in the output above, it means the upload likely failed. You'll need to re-upload it. Ensure that you see a green checkmark next to the uploaded file in the Colab file browser, indicating a successful upload.

Please try restarting your Colab runtime. You can do this by navigating to `Runtime > Restart runtime` in the Colab menu. After the runtime restarts, try uploading the `asl_dataset` again. Make sure to observe the Colab file browser for a green checkmark, which indicates a successful upload.

# Test mobilenet

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, applications
from tensorflow.keras.preprocessing import image_dataset_from_directory

# 1. Load image datasets (assuming directory structure: asl_dataset/class_folder/images.jpg)
# The dataset contains 0-9 and a-z classes, totaling 36 classes.
image_size = (224, 224)  # Updated to 224x224 for MobileNetV2 (better performance with pre-trained weights)
batch_size = 32
num_classes = 36  # 0-9 + a-z

# Load the full dataset and split into training and validation using validation_split
full_dataset = image_dataset_from_directory(
    'asl_dataset',  # Path to the main dataset directory
    image_size=image_size,
    batch_size=batch_size,
    label_mode='categorical',
    validation_split=0.2,  # 20% of data for validation
    subset='both',  # Create both training and validation subsets
    seed=42  # Ensure reproducibility
)

train_dataset = full_dataset[0]
validation_dataset = full_dataset[1]

# Confirm number of classes loaded
print(f"Number of classes detected: {len(train_dataset.class_names)}")
print("Class names:", train_dataset.class_names)

# Optional: Data augmentation to handle variations in hand positions/lighting
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

# 2. Building the Model (Using MobileNetV2 with transfer learning for image classification)
def build_model(num_classes=num_classes):
    # Load pre-trained MobileNetV2 base (feature extractor)
    base_model = applications.MobileNetV2(
        input_shape=image_size + (3,),  # RGB input
        include_top=False,  # Exclude original ImageNet head
        weights='imagenet'  # Use pre-trained weights
    )
    base_model.trainable = False  # Freeze base layers initially for feature extraction

    # Build the full model
    inputs = layers.Input(shape=image_size + (3,))
    x = data_augmentation(inputs)  # Apply augmentation
    x = applications.mobilenet_v2.preprocess_input(x)  # MobileNetV2-specific normalization (replaces Rescaling)
    x = base_model(x, training=False)  # Pass through frozen base
    x = layers.GlobalAveragePooling2D()(x)  # Pool features (better than Flatten for efficiency)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.6)(x)  # Prevent overfitting (kept from your original)
    outputs = layers.Dense(num_classes, activation='softmax')(x)  # Softmax for 36-class output

    model = keras.Model(inputs, outputs)

    model.compile(optimizer='rmsprop',  # Kept from your original; could switch to Adam
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# 3. Training the model (Phase 1: Feature extraction with frozen base)
model = build_model(num_classes=num_classes)
history = model.fit(
    train_dataset,
    epochs=20,
    validation_data=validation_dataset,
    verbose=1
)

# Optional: Phase 2 - Fine-tuning (unfreeze some layers for better adaptation)
# After initial training, unfreeze the base and retrain with low learning rate
base_model = model.get_layer('mobilenetv2_1.00_224')  # Get the base model layer by name
base_model.trainable = True
fine_tune_at = len(base_model.layers) // 2  # Unfreeze the last half of layers
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=1e-5),  # Lower LR to avoid destroying pre-trained weights
              loss='categorical_crossentropy',
              metrics=['accuracy'])

fine_tune_history = model.fit(
    train_dataset,
    epochs=10,  # Fewer epochs for fine-tuning
    validation_data=validation_dataset,
    verbose=1
)

# 4. Evaluate accuracy and other metrics (on the fine-tuned model)
results = model.evaluate(validation_dataset)
print(f"Test Loss: {results[0]}, Test Accuracy: {results[1]}")

# Optional: Save model for inference on new JPEGs
model.save('asl_mobilenetv2_36classes.h5')

# To predict on a new JPEG (same as before, but resize to 224x224 and preprocess):
# new_image = tf.keras.preprocessing.image.load_img('path/to/new_hand_sign.jpg', target_size=(224, 224))
# new_image = tf.keras.preprocessing.image.img_to_array(new_image)
# new_image = applications.mobilenet_v2.preprocess_input(new_image)  # Add this
# new_image = tf.expand_dims(new_image, axis=0)  # Batch dimension
# prediction = model.predict(new_image)
# predicted_class = tf.argmax(prediction, axis=1)

# resnet


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, applications
from tensorflow.keras.preprocessing import image_dataset_from_directory

# 1. Load image datasets (assuming directory structure: asl_dataset/class_folder/images.jpg)
# The dataset contains 0-9 and a-z classes, totaling 36 classes.
image_size = (224, 224)  # 224x224 is standard for ResNet50
batch_size = 32
num_classes = 36  # 0-9 + a-z

# Load the full dataset and split into training and validation using validation_split
full_dataset = image_dataset_from_directory(
    'asl_dataset',  # Path to the main dataset directory
    image_size=image_size,
    batch_size=batch_size,
    label_mode='categorical',
    validation_split=0.2,  # 20% of data for validation
    subset='both',  # Create both training and validation subsets
    seed=42  # Ensure reproducibility
)

train_dataset = full_dataset[0]
validation_dataset = full_dataset[1]

# Confirm number of classes loaded
print(f"Number of classes detected: {len(train_dataset.class_names)}")
print("Class names:", train_dataset.class_names)

# Optional: Data augmentation to handle variations in hand positions/lighting
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomBrightness(factor=0.2),
    layers.RandomContrast(factor=0.2),
    layers.RandomZoom(height_factor=0.15, width_factor=0.15),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
])

# 2. Building the Model (Using ResNet50 with transfer learning)
def build_model(num_classes=num_classes):
    # Load pre-trained ResNet50 base (feature extractor)
    base_model = applications.ResNet50(
        input_shape=image_size + (3,),  # RGB input
        include_top=False,  # Exclude original ImageNet head
        weights='imagenet'  # Use pre-trained weights
    )
    base_model.trainable = False  # Freeze base layers initially for feature extraction

    # Build the full model
    inputs = layers.Input(shape=image_size + (3,))
    x = data_augmentation(inputs)  # Apply augmentation
    x = applications.resnet.preprocess_input(x)  # ResNet50-specific normalization
    x = base_model(x, training=False)  # Pass through frozen base
    x = layers.GlobalAveragePooling2D()(x)  # Pool features
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)  # Prevent overfitting
    outputs = layers.Dense(num_classes, activation='softmax')(x)  # Softmax for 36-class output

    model = keras.Model(inputs, outputs)

    model.compile(optimizer='rmsprop',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# 3. Training the model (Phase 1: Feature extraction with frozen base)
model = build_model(num_classes=num_classes)
history = model.fit(
    train_dataset,
    epochs=20,
    validation_data=validation_dataset,
    verbose=1
)

# Phase 2 - Fine-tuning (unfreeze some layers)
base_model = model.get_layer('resnet50')  # Get the base model layer by name
base_model.trainable = True
fine_tune_at = len(base_model.layers) // 2  # Unfreeze the last half of layers
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

fine_tune_history = model.fit(
    train_dataset,
    epochs=10,  # Fewer epochs for fine-tuning
    validation_data=validation_dataset,
    verbose=1
)

# 4. Evaluate
results = model.evaluate(validation_dataset)
print(f"Test Loss: {results[0]}, Test Accuracy: {results[1]}")

# Save model
model.save('asl_resnet50_36classes.h5')

# Prediction example (for new JPEG)
# new_image = tf.keras.preprocessing.image.load_img('path/to/new_hand_sign.jpg', target_size=(224, 224))
# new_image = tf.keras.preprocessing.image.img_to_array(new_image)
# new_image = applications.resnet.preprocess_input(new_image)
# new_image = tf.expand_dims(new_image, axis=0)
# prediction = model.predict(new_image)
# predicted_class = tf.argmax(prediction, axis=1)

Found 2515 files belonging to 36 classes.
Using 2012 files for training.
Using 503 files for validation.
Number of classes detected: 36
Class names: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 504s 8s/step - accuracy: 0.0685 - loss: 3.6784 - val_accuracy: 0.2565 - val_loss: 2.8858
Epoch 2/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 433s 7s/step - accuracy: 0.1845 - loss: 2.9060 - val_accuracy: 0.3777 - val_loss: 2.3025
Epoch 3/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 434s 7s/step - accuracy: 0.2726 - loss: 2.4406 - val_accuracy: 0.3777 - val_loss: 2.0235
Epoch 4/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 432s 7s/step - accuracy: 0.3356 - loss: 2.1576 - val_accuracy: 0.4911 - val_loss: 1.7065
Epoch 5/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 442s 7s/step - accuracy: 0.3913 - loss: 1.9088 - val_accuracy: 0.4394 - v

# MobileNet50MarchcomparisonModel

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, applications
from tensorflow.keras.preprocessing import image_dataset_from_directory

# 1. Load image datasets (assuming directory structure: asl_dataset/class_folder/images.jpg)
# The dataset contains 0-9 and a-z classes, totaling 36 classes.
image_size = (224, 224)
batch_size = 32
num_classes = 36  # 0-9 + a-z

MODEL_NAME = 'asl_modelDemonstration_v2.keras'

# Load the full dataset and split into training and validation using validation_split
full_dataset = image_dataset_from_directory(
    'asl_dataset',  # Path to the main dataset directory
    image_size=image_size,
    batch_size=batch_size,
    label_mode='categorical',
    validation_split=0.2,  # 20% of data for validation
    subset='both',  # Create both training and validation subsets
    seed=42  # Ensure reproducibility
)

train_dataset = full_dataset[0]
validation_dataset = full_dataset[1]

# Confirm number of classes loaded
print(f"Number of classes detected: {len(train_dataset.class_names)}")
print("Class names:", train_dataset.class_names)

# Optional: Data augmentation to handle variations in hand positions/lighting
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

# 2. Building the Model (Using MobileNetV2 with transfer learning for image classification)
def build_model(num_classes=num_classes):
    # Load pre-trained MobileNetV2 base (feature extractor)
    base_model = applications.MobileNetV2(
        input_shape=image_size + (3,),  # RGB input
        include_top=False,  # Exclude original ImageNet head
        weights='imagenet'  # Use pre-trained weights
    )
    base_model.trainable = False  # Freeze base layers initially for feature extraction

    # Build the full model
    inputs = layers.Input(shape=image_size + (3,))
    x = data_augmentation(inputs)  # Apply augmentation
    x = applications.mobilenet_v2.preprocess_input(x)  # MobileNetV2-specific normalization
    x = base_model(x, training=False)  # Pass through frozen base
    x = layers.GlobalAveragePooling2D()(x)  # Pool features
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.6)(x)  # Prevent overfitting
    outputs = layers.Dense(num_classes, activation='softmax')(x)  # Softmax for 36-class output

    model = keras.Model(inputs, outputs)

    model.compile(optimizer='rmsprop',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# 3. Training the model (Phase 1: Feature extraction with frozen base)
model = build_model(num_classes=num_classes)
history = model.fit(
    train_dataset,
    epochs=20,
    validation_data=validation_dataset,
    verbose=1
)

# Optional: Phase 2 - Fine-tuning (unfreeze some layers for better adaptation)
base_model = model.get_layer('mobilenetv2_1.00_224')
base_model.trainable = True
fine_tune_at = len(base_model.layers) // 2
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

fine_tune_history = model.fit(
    train_dataset,
    epochs=10,
    validation_data=validation_dataset,
    verbose=1
)

# 4. Evaluate accuracy
results = model.evaluate(validation_dataset)
print(f"Test Loss: {results[0]}, Test Accuracy: {results[1]}")

# Save model
model.save(MODEL_NAME)
print(f"Saved {MODEL_NAME}")
print("Download from the Colab files panel on the left.")

Found 2515 files belonging to 36 classes.
Using 2012 files for training.
Using 503 files for validation.
Number of classes detected: 36
Class names: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 15s 100ms/step - accuracy: 0.1128 - loss: 3.2821 - val_accuracy: 0.3022 - val_loss: 2.5872
Epoch 2/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 75ms/step - accuracy: 0.2485 - loss: 2.5950 - val_accuracy: 0.4135 - val_loss: 2.0200
Epoch 3/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 74ms/step - accuracy: 0.3370 - loss: 2.1706 - val_accuracy: 0.6163 - val_loss: 1.5299
Epoch 4/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 75ms/step - accuracy: 0.4274 - loss: 1.8558 - val_accuracy: 0.6640 - val_loss: 1.2371
Epoch 5/20
 3/63 ━━━━━━━━━━━━━━━━━━━━ 3s 57ms/step - accuracy: 0.5104 - loss: 1.6274